# 02 — MinIO RAW Parquet → Apache Iceberg (Nessie catalog)

**RF-02**: read the RAW Parquet from MinIO with **PyArrow** and write it to a
second MinIO bucket as an **Apache Iceberg** table, registered in **Nessie**.

Roles kept explicit because the assignment grades them separately:

- Parquet = file format (physical files)
- Iceberg = table format (organizes those files into a table)
- Nessie  = catalog (knows where the table lives)
- MinIO   = storage (holds both data and metadata)

In [1]:
from __future__ import annotations

import sys

sys.path.insert(0, "/home/jovyan/scripts")

import pyarrow as pa
import pyarrow.parquet as pq

import config as cfg

DLT_INTERNAL_COLUMNS = ("_dlt_load_id", "_dlt_id")


In [2]:
def strip_dlt_columns(table: pa.Table) -> pa.Table:
    present = [name for name in DLT_INTERNAL_COLUMNS if name in table.column_names]
    return table.drop_columns(present) if present else table


cfg.banner("02 | MinIO RAW Parquet -> Iceberg (Nessie catalog)")
print(f"Source   : s3://{cfg.RAW_BUCKET}/{cfg.RAW_DATASET}/{cfg.RAW_TABLE}/")
print(f"Catalog  : {cfg.NESSIE_URI}  (warehouse: {cfg.NESSIE_WAREHOUSE})")
print(f"Table    : {cfg.TABLE_IDENTIFIER}")

filesystem = cfg.minio_filesystem()
prefix = f"{cfg.RAW_BUCKET}/{cfg.RAW_DATASET}/{cfg.RAW_TABLE}"
source_files = cfg.list_parquet(filesystem, prefix)

assert source_files, (
    f"FAIL: no RAW Parquet found under s3://{prefix}/ - run notebook 01 first."
)

print(f"\nRAW files found: {len(source_files)}")
for key in source_files:
    print(f"  {key}")



  02 | MinIO RAW Parquet -> Iceberg (Nessie catalog)
Source   : s3://nyc-taxi-raw/raw/yellow_tripdata/
Catalog  : http://nessie:19120/iceberg  (warehouse: warehouse)
Table    : nyc_taxi.yellow_tripdata_2025_01

RAW files found: 1
  nyc-taxi-raw/raw/yellow_tripdata/1788401857.0521398.08e971fb2c.parquet


Source   : s3://nyc-taxi-raw/raw/yellow_tripdata/
Catalog  : http://nessie:19120/iceberg  (warehouse: warehouse)
Table    : nyc_taxi.yellow_tripdata_2025_01



RAW files found: 1
  nyc-taxi-raw/raw/yellow_tripdata/1788398228.3012493.ff5e43ca4f.parquet


In [3]:
# Schema comes from the first file, minus dlt's bookkeeping columns.
with filesystem.open(source_files[0], "rb") as handle:
    arrow_schema = strip_dlt_columns(
        pq.ParquetFile(handle).schema_arrow.empty_table()
    ).schema


In [4]:
cfg.banner("Nessie | namespace and table")
catalog = cfg.iceberg_catalog()

catalog.create_namespace_if_not_exists((cfg.NESSIE_NAMESPACE,))
print(f"Namespace ready: {cfg.NESSIE_NAMESPACE}")

# Recreating the table keeps this notebook idempotent and guarantees the
# final row count is exact rather than a multiple of the number of runs.
if catalog.table_exists(cfg.TABLE_IDENTIFIER):
    print(f"Dropping existing table {cfg.TABLE_IDENTIFIER}")
    catalog.drop_table(cfg.TABLE_IDENTIFIER)

iceberg_table = catalog.create_table(cfg.TABLE_IDENTIFIER, schema=arrow_schema)
print(f"Table created at: {iceberg_table.location()}")



  Nessie | namespace and table
Namespace ready: nyc_taxi
Dropping existing table nyc_taxi.yellow_tripdata_2025_01
Table created at: s3://nyc-taxi-iceberg/nyc_taxi/yellow_tripdata_2025_01_89f30148-8855-4aa6-be06-8061670c3b93


Namespace ready: nyc_taxi
Dropping existing table nyc_taxi.yellow_tripdata_2025_01


Table created at: s3://nyc-taxi-iceberg/nyc_taxi/yellow_tripdata_2025_01_4ba6d1f7-dd1d-45d0-9e42-830d1b373f5b


In [5]:
cfg.banner("Writing data")
written = 0
for key in source_files:
    with filesystem.open(key, "rb") as handle:
        parquet_file = pq.ParquetFile(handle)
        for batch in parquet_file.iter_batches(batch_size=cfg.BATCH_ROWS):
            chunk = strip_dlt_columns(pa.Table.from_batches([batch]))
            iceberg_table.append(chunk)
            written += chunk.num_rows
            print(f"  appended {written:,} rows", flush=True)



  Writing data
  appended 250,000 rows
  appended 500,000 rows
  appended 750,000 rows
  appended 1,000,000 rows
  appended 1,250,000 rows
  appended 1,500,000 rows
  appended 1,750,000 rows
  appended 2,000,000 rows
  appended 2,250,000 rows
  appended 2,500,000 rows
  appended 2,750,000 rows
  appended 3,000,000 rows
  appended 3,250,000 rows
  appended 3,475,226 rows


  appended 250,000 rows


  appended 500,000 rows


  appended 750,000 rows


  appended 1,000,000 rows


...

  appended 2,500,000 rows


  appended 2,750,000 rows


  appended 3,000,000 rows


  appended 3,250,000 rows


  appended 3,475,226 rows


In [6]:
cfg.banner("Verification")
iceberg_table.refresh()
scanned = iceberg_table.scan().to_arrow().num_rows
print(f"Rows in Iceberg table : {scanned:,}")
print(f"Snapshots             : {len(iceberg_table.metadata.snapshots)}")
print(f"Namespaces in Nessie  : {catalog.list_namespaces()}")
print(f"Tables in namespace   : {catalog.list_tables(cfg.NESSIE_NAMESPACE)}")

data_files = cfg.list_parquet(filesystem, cfg.ICEBERG_BUCKET)
metadata_files = sorted(filesystem.glob(f"{cfg.ICEBERG_BUCKET}/**/metadata/*"))
print(f"\nData files in s3://{cfg.ICEBERG_BUCKET}     : {len(data_files)}")
print(f"Metadata files in s3://{cfg.ICEBERG_BUCKET} : {len(metadata_files)}")

assert scanned == cfg.EXPECTED_ROWS, (
    f"expected {cfg.EXPECTED_ROWS:,} rows, found {scanned:,}"
)
print("\nOK - Iceberg table matches the expected row count.")



  Verification
Rows in Iceberg table : 3,475,226
Snapshots             : 1
Namespaces in Nessie  : [('nyc_taxi',)]
Tables in namespace   : [('nyc_taxi', 'yellow_tripdata_2025_01')]

Data files in s3://nyc-taxi-iceberg     : 42
Metadata files in s3://nyc-taxi-iceberg : 129

OK - Iceberg table matches the expected row count.


Rows in Iceberg table : 3,475,226
Snapshots             : 1
Namespaces in Nessie  : [('nyc_taxi',)]
Tables in namespace   : [('nyc_taxi', 'yellow_tripdata_2025_01')]

Data files in s3://nyc-taxi-iceberg     : 28
Metadata files in s3://nyc-taxi-iceberg : 86

OK - Iceberg table matches the expected row count.
